## Reproducing the defect pattern by means of the Jacobian - _A sanity check_

Here we follow the classic "Boolean Derivatives on Cellular Automata" by Gérard Vichniac (1990).

consider an elementary cellular automaton of $N$ cells and periodic boundary conditions. The local update rule
$$
f : \{0,1\}^n \mapsto \{0,1\}
$$
maps a neighbourhood configuration of size $n=3$ to either $0$ or $1$. As an example, consider rule $54$, whose binary expression is $00110110_2$. Following one of the [Tables of Cellular Automaton Properties](https://content.wolfram.com/sw-publications/2020/07/cellular-automaton-properties.pdf), this translates to the Boolean expression
$$
f_{(54)}(x_{i-1}, x_{i}, x_{i+1}) = (\bar{x}_{i-1} x_i \bar{x}_{i+1}) + (x_{i-1} \bar{x}_i) + (\bar{x}_{i} x_{i+1}),
$$
where $x_i$ is the state of cell $i$ at some time step, and $f(x_{i-1}, x_i, x_{i+1})$ is the state of that cell in the subsequent time step. A bar ($\bar{x}$) denotes the binary complement ($\bar{1} = 0, \bar{0} = 1$). The function $f$ induces a global update function
$$
F : \{0,1\}^N \mapsto \{0,1\}^N,
$$
where we say $F_i(\vec{x}) = F_i(x_1, \ldots, x_N) = f(x_{i-1}, x_{i}, x_{i+1})$.

The Jacobian $J$ is a matrix with all partial derivatives of a vector-valued function. Its elements are therefore
$$
J_{ij} = F'_{ij} = F_i(x_1, \ldots, x_j, \ldots, N) \oplus F_i(x_1, \ldots, \bar{x}_j, \ldots, N).
$$
Here the sum modulo 2 is the Boolean interpretation of a derivative. We can interpret $J_{ij}$ as the answer to the question "what happens to cell $i$ in the next time step if we change the value of cell $j$ in the current time step":
- $J_{ij}=0$: cell $i$ stays the same
- $J_{ij}=1$: cell $i$ changes to its binary complement

Because the state vector $\vec{x}$ generally changes over time, the Jacobian is also time-dependent. Because for an ECA a cell can only communicate with its direct neighbours in a single time step, all $J_{ij}$ for which $i \notin \{j-1, j, j+1\}$ (modulo $N$ for periodic boundary conditions) are $0$, such that $J$ is tri-diagonal with the exception of the off-diagonal corner elements (periodic boundary conditions). Concretely:
$$
J_{ij} = f(x_{i-1}, x_i, x_{i+1}) \oplus \begin{cases}
f(\bar{x}_{i-1}, x_i, x_{i+1}) \text{ if } j=i-1, \\
f(x_{i-1}, \bar{x}_i, x_{i+1}) \text{ if } j=i, \\
f(x_{i-1}, x_i, \bar{x}_{i+1}) \text{ if } j=i+1,
\end{cases}
$$
and $J_{ij}=0$ in all other cases. Index $N+1$ is identified with index $1$, and index $0$ is identified with index $N$. For our example (rule $54$), this results in a Jacobian
$$
J = \begin{pmatrix}
J_{11} & J_{12} & \cdots & J_{1N} \\
J_{21} & J_{22} & \cdots & J_{2N} \\
\vdots & \vdots & \ddots & \vdots \\
J_{N1} & J_{N2} & \cdots & J_{NN}
\end{pmatrix} = 
 \begin{pmatrix}
1 & \bar{x}_{N} &  &  &  & \bar{x}_{2} \\
\bar{x}_{3} & 1 & \bar{x}_1 &  &  &  \\
 & \bar{x}_{4} & 1 & \bar{x}_2 &  &  \\
 &  &  & \ddots &  &  \\
 &  &  & \bar{x}_N & 1 & \bar{x}_{N-2} \\
\bar{x}_{N-1} &  &  &  & \bar{x}_{1} & 1
\end{pmatrix}
$$
and $J_{ij}=0$ in all empty entries.

In continuous analysis, we write
$$
F(\vec{x} + \delta\vec{x}) - F(\delta\vec{x}) = F'(\vec{x}) \delta\vec{x} + \mathcal{O}(\delta\vec{x}) = J(\vec{x})\delta\vec{x} + \mathcal{O}(\delta\vec{x}).
$$
The last term vanishes for $\delta\vec{x} \rightarrow 0$. In our case, however, we cannot meaningfully perform the limit $\delta\vec{x} \rightarrow 0$, because the minimum "defect" has unity length ($\delta\vec{x} = \vec{e_i}$ for a defect in cell $i$). In terms of our ECA, this becomes
$$
F(\vec{x} + \delta\vec{x}) \oplus F(\delta\vec{x}) = J(\vec{x}) \otimes \delta\vec{x} \oplus \mathcal{O}(\delta\vec{x}).
$$
where $A \otimes \vec{v} = (A \times \vec{v}) \mod 2 = \vec{w}$ with elements $w_i = \bigoplus_{j=1}^N A_{ij}v_j$. The claim is that the final term should vanish for sufficiently small defects. Let's try to verify this first.

In [ ]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
import cellpylib as cpl

plt.rcParams['text.usetex'] = True

In [ ]:
# define Jacobian based on the formula above
def jacobian_eca(config, rule):
    N = len(config)
    J = np.zeros(shape=(N,N), dtype=int)
    left_defect = np.array([1, 0, 0])
    central_defect = np.array([0, 1, 0])
    right_defect = np.array([0, 0, 1])
    for i in range(N):
        # take neighbourhood and calculate next state
        nbh = np.array([config[(i-1)%N], config[i], config[(i+1)%N]])
        next_state = cpl.nks_rule(nbh, rule)
        # perturb neighbourhood in three places
        left_perturbed = np.bitwise_xor(nbh, left_defect)
        central_perturbed = np.bitwise_xor(nbh, central_defect)
        right_perturbed = np.bitwise_xor(nbh, right_defect)
        # calculate next state from perturbed neighboorhood
        next_lp = cpl.nks_rule(left_perturbed, rule)
        next_cp = cpl.nks_rule(central_perturbed, rule)
        next_rp = cpl.nks_rule(right_perturbed, rule)
        # fill in non-trivial Jacobian values
        J[i, (i-1)%N] = np.bitwise_xor(next_state, next_lp)
        J[i, i] = np.bitwise_xor(next_state, next_cp)
        J[i, (i+1)%N] = np.bitwise_xor(next_state, next_rp)
    return J

# J = np.zeros(shape=(N,N), dtype=int)
# fill in values found in Vichniac Table 1

# for i in range(N): # rule 54
#     J[i,(i-1)%N] = 1-init_config[(i+1)%N]
#     J[i,i] = 1
#     J[i,(i+1)%N] = 1-init_config[(i-1)%N]

# for i in range(N): # rule 110 # WRONG in Vichniac's paper!!
#     J[i,(i-1)%N] = (1-init_config[i])*init_config[(i+1)%N]
#     J[i,i] = np.min((1,init_config[(i-1)%N] + (1-init_config[(i+1)%N])))
#     J[i,(i+1)%N] = np.min((1,init_config[(i-1)%N] + init_config[i]))

# for i in range(N): # rule 60
#     J[i,(i-1)%N] = 1
#     J[i,i] = 1
#     J[i,(i+1)%N] = 0

# for i in range(N): # rule 73
#     J[i,(i-1)%N] = np.min((1,init_config[i] + (1-init_config[(i+1)%N])))
#     J[i,i] = np.min((1,(1-init_config[(i-1)%N]) + (1-init_config[(i+1)%N])))
#     J[i,(i+1)%N] = np.min((1,(1-init_config[(i-1)%N])+init_config[i]))

In [ ]:
# verify over 100 cells (definitely enough)
N = 10
samples_per_rule = 10
error_counter = 0
OK_rules = []

for rule in range(256):
    OK_rule = 1
    print(f"Checking rule {rule}.", end='\r')
    # add a very simple defects
    init_defect = np.zeros(N, dtype=int)
    for _ in range(samples_per_rule):
        init_defect[0] = 1 # np.random.randint(2)
        init_config = np.random.randint(2,size=N)
        init_config_def = (init_config + init_defect) % 2
        # calculate next configuration with and without defect
        next_config = cpl.evolve(init_config[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        next_config_def = cpl.evolve(init_config_def[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        # calculate Jacobian
        J = jacobian_eca(init_config, rule)
        # see whether the LHS and RHS of the expansion are identical
        LHS = (next_config_def + next_config) % 2
        RHS = np.matmul(J,init_defect)
        if not (LHS==RHS).all():
            OK_rule = 0
    OK_rules.append(OK_rule)
if (np.array(OK_rules) == 1).all():
    print("\nThe equality holds for all rules that we checked after introducing a minimal defect in a single cell.")

# verify with defects in non-connected places
error_counter = 0
OK_rules = []

for rule in range(256):
    OK_rule = 1
    print(f"Checking rule {rule}.", end='\r')
    # add a very simple defects
    init_defect = np.zeros(N, dtype=int)
    for _ in range(samples_per_rule):
        init_defect[0] = 1 # np.random.randint(2)
        init_defect[3] = 1
        init_config = np.random.randint(2,size=N)
        init_config_def = (init_config + init_defect) % 2
        # calculate next configuration with and without defect
        next_config = cpl.evolve(init_config[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        next_config_def = cpl.evolve(init_config_def[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        # calculate Jacobian
        J = jacobian_eca(init_config, rule)
        # see whether the LHS and RHS of the expansion are identical
        LHS = (next_config_def + next_config) % 2
        RHS = np.matmul(J,init_defect)
        if not (LHS==RHS).all():
            OK_rule = 0
    OK_rules.append(OK_rule)
if (np.array(OK_rules) == 1).all():
    print("\nThe equality holds for all rules that we checked after introducing a minimal defect in cells that do not have an overlapping neighbourhood.") 

This seems to work out. The equality
$$
F(\vec{x} + \delta\vec{x}) \oplus F(\vec{x}) = J(\vec{x}) \otimes \delta\vec{x}.
$$
holds as long as $\mathcal{O}(\delta\vec{x}) = 0$: the Boolean derivative "extracts the linear part of arbitrary rules" (Vichniac). This occurs in general when the introduced defect does not affect more than one neighbourhood simultaneously. Let's inspect what happens when this condition is not met.

In [ ]:
# verify over 10 cells
N = 10
samples_per_rule = 20
error_counter = 0
OK_rules = []

for rule in range(256):
    OK_rule = 1
    print(f"Checking rule {rule}.", end='\r')
    # add a very simple defects
    init_defect = np.zeros(N, dtype=int)
    for _ in range(samples_per_rule):
        # add more than one defect
        init_defect[0] = 1
        init_defect[1] = 1
        init_defect[2] = 1
        init_config = np.random.randint(2,size=N)
        init_config_def = (init_config + init_defect) % 2
        # calculate next configuration with and without defect
        next_config = cpl.evolve(init_config[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        next_config_def = cpl.evolve(init_config_def[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[-1].astype(int)
        # calculate Jacobian
        J = jacobian_eca(init_config, rule)
        # see whether the LHS and RHS of the expansion are identical
        LHS = (next_config_def + next_config) % 2
        RHS = np.matmul(J,init_defect)
        if not (LHS==RHS).all():
            OK_rule = 0
    OK_rules.append(OK_rule)

OK_rules, = np.where(OK_rules)
print(f"When introducing a defect in cells with overlapping neighbourhoods, the equality no longer holds for {256-len(OK_rules)} rules.\nDespite the larger defect, the equality still holds for rules {OK_rules}.\nNote that the particular list of rules depends on the specific nature of the multiple-cell defect.")

Mathematically, the equality works when all elements are equal:
$$
\forall i \in \{1, \ldots, N\}: F_i(\vec{x} + \delta\vec{x}) \oplus F_i(\vec{x}) =
f(x_{i-1} \oplus \delta x_{i-1}, x_i \oplus \delta x_i, x_{i+1} \oplus \delta x_{i+1}) \oplus f(x_{i-1}, x_i, x_{i+1})
= J_{i(i-1)}\delta x_{i-1} \oplus J_{ii}\delta x_i \oplus J_{i(i+1)}\delta x_{i+1}
$$
This equality obviously works out when $\delta \vec{x}$ is a one-hot vector (all $0$'s except a single $1$), because then we recover precisely the definition of the Boolean derivative that is contained in the Jacobian. More generally, however, this explicit equation demonstrates mathematically the two observations made above:
1. Defects in cells with non-overlapping neighbourhoods still respect the linearisation
2. Defects in cells with overlapping neighbourhoods no longer do in general, but by following the equation above, we could deduce for which rules and for which configurations it does

In brief, though: **linearisation only works for minimal defects!**

A special case in which the equations above could be worked out further are linear ECAs. Linearity for a continuous function $g$ implies that $g(a(x+y)) = ag(x) + ag(y)$.
In mathematical terms, an elementary cellular automaton is linear if its rule can be represented by a linear polynomial over the finite field $\mathbb{F}_2$ (which consists of the binary values 0 and 1). The XOR operation corresponds to addition in this field, and the AND operation corresponds to multiplication.

Linearity then implies
$$
x_i^{t+1} = f(x_{i-1}^t, x_{i}^t, x_{i-1}^t) = a x_{i-1}^t \oplus b x_{i}^t \oplus c x_{i+1}^t, \text{ for } a, b, c \in \{0, 1\}.
$$
Say the binary representation of the ECA rule is $b_7 b_6 b_5 b_4 b_3 b_2 b_1 b_0$.  For a linear rule, the equations above should hold for each of the eight neighbourhoods, and for each of the eight combinations $\{a, b, c\} \in \{0,1\}^3$, so we end up with equations
$$
\begin{cases}
b_0 = 0,\\
b_1 = c,\\
b_2 = b,\\
b_3 = b \oplus c,\\
b_4 = a,\\
b_5 = a \oplus c, \\
b_6 = a \oplus b, \\
b_7 = a \oplus b \oplus c
\end{cases}
$$
This presents us with eight such rules:
$$
\begin{cases}
\{a=0, b=0, c=0\} : 00000000_2 = 0, \\
\{a=0, b=0, c=1\} : 10101010_2 = 170, \\
\{a=0, b=1, c=0\} : 11001100_2 = 204, \\
\{a=0, b=1, c=1\} : 01100110_2 = 102, \\
\{a=1, b=0, c=0\} : 11110000_2 = 240, \\
\{a=1, b=0, c=1\} : 01011010_2 = 90, \\
\{a=1, b=1, c=0\} : 00111100_2 = 60, \\
\{a=1, b=1, c=1\} : 10010110_2 = 150.
\end{cases}
$$
Naturally, their complement and their left-right symmetric brothers are linear as well.

**QUESTION** but why can't I calculate them then? Is this statement true?

In [ ]:
# function for finding all equivalent rules
def equivalent_set(rule):
    binary = format(rule, f'08b')
    binary_array = np.array(list(binary), dtype=int)
    powers_of_2 = np.array([2**i for i in range(7,-1,-1)])
    # find complement
    comp_array = 1 - binary_array
    comp_rule = np.dot(comp_array, powers_of_2)
    # find mirror
    mirror_array = binary_array.copy()
    mirror_array[2] = binary_array[4]
    mirror_array[4] = binary_array[2]
    mirror_array[3] = binary_array[6]
    mirror_array[6] = binary_array[3]
    mirror_rule = np.dot(mirror_array, powers_of_2)
    # find complement of mirror
    comp_mirror_array = 1 - mirror_array
    comp_mirror_rule = np.dot(comp_mirror_array, powers_of_2)
    return set([rule, comp_rule, mirror_rule, comp_mirror_rule])

unique_lin_rules = [0, 170, 204, 102, 240, 90, 60, 150]
linear_rules = set()
for rule in unique_lin_rules:
    linear_rules = linear_rules.union(equivalent_set(rule))
linear_rules = np.sort(np.array(list(linear_rules)))

print(f"There are {len(linear_rules)} linear ECA rules: {linear_rules}.")

In the context of the equation above (and also relevant for Lyapunov exponents), these rules are special because their Jacobian is a constant. It is independent of the configuration:
$$
J(\vec{x}^t) = J.
$$

We could continue further, making a mathematical derivation of which rules propagate the errors perfectly. However, much of this has already been done by Vichniac, amongst others. What matters for our purposes is that, because the expansion $F(\vec{x} + \delta\vec{x}) \oplus F(\vec{x}) = J(\vec{x}) \otimes \delta\vec{x}$ only holds for small defects, it cannot generally be used to predict configurations on later time steps. More concretely, say that $\delta\vec{x}^{t}$ is a unit defect introduced at time $t$. Then
$$
F(\vec{x}^t \oplus \delta\vec{x}^{t}) \oplus F(\vec{x}^{t}) = J(\vec{x}^{t}) \otimes \delta\vec{x}^{t} = \delta\vec{x}^{t+1}
$$
is the difference pattern between the global update from the perturbed and unperturbed initial configuration $\vec{x}$. This difference pattern could of course be used to reconstruct the global evolution from the initially perturbed state:
$$
F(\vec{x}^{t} \oplus \delta\vec{x}^{t}) = F(\vec{x}^{t}) \oplus \delta\vec{x}^{t+1}.
$$
However, this _not_ in general true for the next time step:
$$
F(F(\vec{x}^{t} \oplus \delta\vec{x}^{t})) \neq F(F(\vec{x}^{t})) \oplus J(F(\vec{x}^{t})) \otimes \delta\vec{x}^{t+1},
$$
because $\delta\vec{x}^{t+1}$ is not guaranteed to be $\vec{0}$ or a unit defect.

## A conceptual problem with updating the Jacobian

In continuous analysis, the Jacobian consists of derivatives that are defined by taking the limit of a infinitesimal initial defect. In the case of a discrete dynamical system, no such limit can meaningfully be made. This circumvented by making use of a Boolean derivative, which incorporates the finitude of the defect.

However, when a further claim is being made, I feel like some issues do arise. In continuous analysis, the Jacobian can be used to trace the evolution of a sphere of perturbations along the tangent plane at $\vec{x}$:
$$
\dot{\mathbf{Y}} = \mathbf{JY}.
$$
In terms of CA, it is claimed that this translates to 
$$
\mathbf{Y}^{t+1} = \mathbf{J}(\vec{x}^t) \ldots \mathbf{J}(\vec{x}^0) \mathbf{Y}^0.
$$
I do not really understand this:
1. Why has the differential equation turned into a regular equation?
2. If we are following the perturbations in tangent space, why do we need to keep inputting information from configuration space? It feels like we are mixing up information here.

## A mistake in the 1990 Vichniac paper

In Vichniac's 1990 paper "Boolean derivatives on cellular automata", Table 1 contains gradients of peripheral one-dimensional cellular automata:
$$
\nabla F_i = \left( \frac{\partial F_i}{\partial x_{j-1}}, \frac{\partial F_i}{\partial x_{j}}, \frac{\partial F_i}{\partial x_{j+1}}\right) = \left(J_{i(j-1)}, J_{ij}, J_{i(j+1)} \right),
$$
i.e. the row elements of the Jacobian $\mathbf{J}$. Recall that only the elements $J_{ii}$ and $J_{i(i\pm1)}$ can be non-zero. Recall, in addition, that generally the elements of $\mathbf{J}$ depend on the configuration $\vec{x}$, except for linear CAs. For rule $110$ they _do_ depend on the configuration, and Vichniac claims that
$$
\begin{cases}
J_{i(i-1)} = \bar{x}_i x_{i+1},\\
J_{ii} = x_{i-1} + \bar{x}_{i+1},\\
J_{i(i+1)} = x_{i-1} + x_i,
\end{cases}
$$
where $+$ is the OR operator ($1+1=1$). However, the definition of the Boolean derivative tells us that
$$
\begin{cases}
J_{i(i-1)} = f(x_{i-1}, x_i, x_{i+1}) \oplus f(\bar{x}_{i-1}, x_i, x_{i+1}),\\
J_{ii} = f(x_{i-1}, x_i, x_{i+1}) \oplus f(x_{i-1}, \bar{x}_i, x_{i+1}),\\
J_{i(i+1)} = f(x_{i-1}, x_i, x_{i+1}) \oplus f(x_{i-1}, x_i, \bar{x}_{i+1}),
\end{cases}
$$
Let us quickly list these side-by-side for all neighbourhoods in $\{0,1\}^3$

In [ ]:
def J_vichniac_110(nbh):
    # filled out from the table. THIS IS WRONG
    left = nbh[0]; centre = nbh[1]; right = nbh[2]
    J_left = int((not centre) and right)
    J_centre = int(left or (not right))
    J_right = int(left or centre)
    J = [J_left, J_centre, J_right]
    return J

def J_vichniac_72(nbh):
    # correct
    left = nbh[0]; centre = nbh[1]; right = nbh[2]
    J_left = int(centre)
    J_centre = int((left and (not right)) or ((not left) and right))
    J_right = int(centre)
    J = [J_left, J_centre, J_right]
    return J

def J_vichniac_23(nbh):
    # correct
    left = nbh[0]; centre = nbh[1]; right = nbh[2]
    J_left = int((centre and (not right)) or ((not centre) and right))
    J_centre = int((left and (not right)) or ((not left) and right))
    J_right = int((left and (not centre)) or ((not left) and centre))
    J = [J_left, J_centre, J_right]
    return J

def J_def(nbh, rule):
    next_state = cpl.nks_rule(nbh, rule)
    next_state_left = cpl.nks_rule(np.bitwise_xor(np.array([1,0,0]), np.array(nbh)), rule)
    next_state_centre = cpl.nks_rule(np.bitwise_xor(np.array([0,1,0]), np.array(nbh)), rule)
    next_state_right = cpl.nks_rule(np.bitwise_xor(np.array([0,0,1]), np.array(nbh)), rule)
    J_left = np.bitwise_xor(next_state, next_state_left)
    J_centre = np.bitwise_xor(next_state, next_state_centre)
    J_right = np.bitwise_xor(next_state, next_state_right)
    J = [J_left, J_centre, J_right]
    return J

rule = 110
print(f"Rule {rule}.")
for digit in range(8):
    nbh = np.array(list(format(digit, f'03b'))).astype(int)
    print(f"{nbh}: {J_vichniac_110(nbh)} (Vichniac)")
    print(f"       : {J_def(nbh, rule)} (definition)")

Clearly this is wrong. Let us calculate the correct expression. In Boolean terms, we find
$$
f_{(110)}(x_{i-1}, x_i, x_{i+1}) = (\bar{x}_{i-1} x_i) + (x_i \bar{x}_{i+1}) + (\bar{x}_i x_{i+1})
$$
Applying the definition of the Boolean derivatives, we then find
$$
\begin{cases}
J_{i(i-1)} = x_i x_{i+1},\\
J_{ii} = x_{i-1} + \bar{x}_{i+1},\\
J_{i(i+1)} = x_{i-1} + \bar{x}_i
\end{cases}
$$

In [ ]:
def J_vichniac_110_new(nbh):
    # calculated myself
    left = nbh[0]; centre = nbh[1]; right = nbh[2]
    J_left = int(centre and right)
    J_centre = int(left or (not right))
    J_right = int(left or (not centre))
    J = [J_left, J_centre, J_right]
    return J

rule = 110
print(f"Rule {rule}.")
for digit in range(8):
    nbh = np.array(list(format(digit, f'03b'))).astype(int)
    print(f"{nbh}: {J_vichniac_110_new(nbh)} (Vichniac corrected)")
    print(f"       : {J_def(nbh, rule)} (definition)")

Now it's correct.

## Previous stuff

Below are some of the notes I made earlier, but they do not contribute to the main story.

In [ ]:
# calculate the Jacobian
def jacobian(config, rule):
    N = config.shape[-1]
    config_next = cpl.evolve(init_config, timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[1]
    # make matrix with N rows of the same state array
    config_all = np.tile(config, (N,1))
    # calculate unperturbed next-timestep state vector (N times, all identical)
    config_next_all = np.tile(config_next, (N,1))

    # make matrix with all possible single-defect perturbations
    config_defect_all = np.abs(config_all - np.diag(np.ones(N)))

    # calculate perturbed next-timestep state vector
    # NOTE: this is not programmed efficiently
    config_defect_next_all = np.zeros(shape=(N,N))
    for i, config_defect in enumerate(config_defect_all):
        config_defect_next = cpl.evolve(config_defect[np.newaxis,:], timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))[1]
        config_defect_next_all[i] = config_defect_next
    
    # compile Jacobian with modulo-2 operator (which is basically the Boolean derivative)
    # NOTE: make sure this does not have to be the transposed matrix
    J = np.mod(config_next_all + config_defect_next_all, 2)
    return J

fig, axs = plt.subplots(1,4, figsize=(12,3))

# pick a rule and the number of cells
rule = np.random.randint(256)
width = 50
for i, ax in enumerate(axs):
    # load an initial configuration
    init_config = np.random.randint(2, size=(1,width))
    J = jacobian(init_config, rule)
    ax.imshow(J, cmap='Greys')
    ax.set_title(f"Init. config {i+1}")
_=fig.suptitle(f"Rule {rule}")

print(f"Note that the particular Jacobian depends not only on the rule, but also at the initial configuration.")

In [ ]:
width=100 # let's take this big enough to make sure that we leave nothing to chance
init_config = np.random.randint(2, size=(1,width))
rules = np.arange(256)

identicals = []
for rule in rules:
    print(f"Working on rule {rule}   ", end='\r')
    # calculate the next time step with cellpylib
    configs = cpl.evolve(init_config, timesteps=2, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))
    next_config = configs[1]

    # calculate the Jacobian
    J = jacobian(init_config, rule)

    # using the Jacobian, come up with the next time step
    next_config_J = np.mod(np.matmul(J, init_config[0]),2).astype(int)

    # compare both outputs
    identical = (next_config_J == next_config).all()
    identicals.append(identical)


In [ ]:
OK_rules, = np.where(np.array(identicals)==True)
print(f"All rules that obey the Jacobian multiplication are {OK_rules}.")

That is to say: **The expression $\vec{s}^t = J^{t-1} \otimes \vec{s}^{t-1}$ is not generally true!**

Now, to be fair, that is not necessarily what is being claimed in the papers. What _is_ being claimed, however, is the propagation of the initial perturbation 'sphere':

$$
Y^{t+1} = J(\vec{s}^t) \times \cdots \times J(\vec{s}^0) Y^0,
$$

where $Y^0$ is the unit diagonal, and now the $\times$ operator is not modulo-2. What we want to verify is whether it is possible to recover the actual state of the ECA by means of the matrix $Y^{t+1}$. Mathematically, the claim would be:

$$
{}^j\vec{s}^{t+1} = \vec{s}^{t+1} \oplus Y^{t+1}_j = \vec{s}^{t+1} \oplus (J(\vec{s}^t) \times \cdots \times J(\vec{s}^0) \times Y^0)_j,
$$

where ${}^j\vec{s}^{t+1}$ is the state vector at time $t$ for which cell $j$ was perturbed at time $t=0$. $Y^{t+1}_j$ is the $j$-th row of the perturbation 'sphere' after $t+1$ time steps, i.e. the perturbation in each of the dimensions (cells) caused by an original defect in cell $j$. The $\oplus$ operator again represent summation modulo 2, in an element-wise fashion.

My suspicion is that this will not work, because we would be confusing tangent space (in which the perturbation sphere $Y$ 'lives') and configuration space (in which the state vector $\vec{s}$ lives).

In [ ]:
width = 100
T = 2
rule = np.random.randint(256)

# open the matrix
Y = np.diag(np.ones(width))

# take an initial configuration and calculate the Y^t+1 matrix
init_config = np.random.randint(2, size=(1,width))
configs = cpl.evolve(init_config, timesteps=T, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))

# iterate over the time steps
for t in range(T-1):
    J = jacobian(configs[t], rule)
    Y = np.matmul(J, Y)

In [ ]:
# take the same initial configuration, but with one defect in cell j
j = np.random.randint(width)
init_config_defect = init_config.copy()
init_config_defect[0,j] = 1-init_config_defect[0,j]

# evolve the CA from this defected configuration
configs_defect = cpl.evolve(init_config_defect, timesteps=T, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))

# check the supposed equality
(configs_defect[-1] == np.mod(configs[-1] + Y[j], 2)).all()

In [ ]:
# do this for all rules
rules = np.arange(256)
Ts = np.arange(1,8) # running until T=8 requires ~30 minutes
identicals_per_T = []
correct_cells_per_T = []
for T in Ts:
    identicals = []
    correct_cells = []
    for rule in rules:
        print(f"Working on rule {rule} over {T} time steps.   ", end='\r')
        # open the matrix
        Y = np.diag(np.ones(width))

        # take an initial configuration and calculate the Y^t+1 matrix
        init_config = np.random.randint(2, size=(1,width))
        configs = cpl.evolve(init_config, timesteps=T, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))

        # iterate over the time steps
        for t in range(T-1):
            J = jacobian(configs[t], rule)
            Y = np.matmul(J, Y)

        # take the same initial configuration, but with one defect in cell j
        j = np.random.randint(width)
        init_config_defect = init_config.copy()
        init_config_defect[0,j] = 1-init_config_defect[0,j]

        # evolve the CA from this defected configuration
        configs_defect = cpl.evolve(init_config_defect, timesteps=T, apply_rule=lambda n, c, t: cpl.nks_rule(n, rule))

        # check the supposed equality
        identical = (configs_defect[-1] == np.mod(configs[-1] + Y[j], 2)).all()
        identicals.append(identical)

        # number of cells in the correct state
        correct_cell = np.sum(configs_defect[-1] == np.mod(configs[-1] + Y[j], 2))
        correct_cells.append(correct_cell)

    identicals_per_T.append(np.array(identicals))
    correct_cells_per_T.append(np.array(correct_cells))

In [ ]:
# show the number of rules that still match up in configuration space after t time steps
fig, axs = plt.subplots(2,1,figsize=(10,7), sharex=True)

total_identities = [np.sum(identicals_per_T[i]) for i in range(T)]

axs[0].plot(total_identities)
axs[0].set_xticks(range(T))
axs[0].set_yticks([0, 64, 128, 196, 256])
axs[0].set_xlabel("Time steps after initial configuration")
axs[0].set_ylabel("Number of rules with perfect match")
axs[0].set_title(r"Success rate of predicting the defect pattern with the Jacobian. ${}^j\vec{s}^{t+1} \stackrel{?}{=} \vec{s}^{t+1} \oplus Y^{t+1}_j$")

arrowprops = dict(facecolor='black', shrink=0.02, width=1, headwidth=5)

for i in range(2,T):
    annotation_text = np.where(identicals_per_T[i])[0]
    point_x = i; point_y = total_identities[point_x]

    axs[0].annotate(
        annotation_text,  # Text to display
        rotation=0,
        horizontalalignment='left',
        xy=(point_x, point_y),  # Point to annotate
        xytext=(point_x-2.5, point_y + 48*(i-1)-32),  # Position of the text
        arrowprops=arrowprops,  # Properties of the arrow
        bbox=dict(boxstyle='round,pad=0.5', edgecolor='black', facecolor='lightyellow'),
        fontsize=8
    )

# in the bottom plot, only plot the imperfect rules
all_perfect_rules = np.arange(256)
for i in range(T):
    all_perfect_rules = np.intersect1d(all_perfect_rules, np.where(identicals_per_T[i])[0])

print(f"All 'perfect' rules: ", all_perfect_rules)

correct_cells_per_T_imperfect = [np.delete(correct_cells_per_T[i], all_perfect_rules) for i in range(T)]

# and for all time steps, show how many cells are mismatching in configuration space vs. tangent space
mins = np.array([np.min(correct_cells_per_T_imperfect[i]) for i in range(T)])
maxs = np.array([np.max(correct_cells_per_T_imperfect[i]) for i in range(T)])
means = np.array([np.mean(correct_cells_per_T_imperfect[i]) for i in range(T)])
stdevs = np.array([np.std(correct_cells_per_T_imperfect[i]) for i in range(T)])

axs[1].plot(means, label='mean over all rules')
axs[1].fill_between(range(T), means-stdevs, means+stdevs, alpha=.5, label='standard deviation')

axs[1].plot(np.array(correct_cells_per_T)[:,0], color='olive', alpha=0.03, label='single rule')
axs[1].plot(np.array(correct_cells_per_T)[:,1:], color='olive', alpha=0.03)

axs[1].plot([0, T-1], [50, 50], 'k--', lw=1, label='expected random value')

axs[1].legend(ncols=2)

axs[1].set_xticks(range(T))
axs[1].set_yticks([0, 20, 40, 60, 80, 100])
axs[1].set_xlabel("Time steps after initial configuration")
axs[1].set_ylabel("Number of cells in the correct state")
num_of_imperfect = 256-len(all_perfect_rules)
axs[1].set_title(f"Correct cell states for all {num_of_imperfect} imperfect rules, after predicting the defect pattern with the Jacobian")

Some observations are quite striking:
1. Some rules will always generate a correct difference pattern using the Jacobian technique. In other words, hese rules appear to accommodate an equivalence between "tangent space modulo 2" and configuration space.
2. Strikingly, the set of rules that generate this equivalence is not constant. This may be the result of chance, but considering the large size of the CA, such mere probabilities are expected to be small.
3. Some usual suspects do stand out, however: rules 0, 4, 68, 204, ... These should be investigated further with respect to their genotype.
4. The mean number of cells that are correctly predicted using the Jacobian technique appears to converge fast, and is clearly higher than one would expect from pure chance alone.
5. This is perhaps not the best measure, however, as we also see that individual rules (olive-coloured) have quite a large variance for subsequent time steps

**Question**: if the time step at t+1 can be calculated perfectly, why can't this be done iteratively from that point onward? A Taylor expansion is not perfect because it stops being perfect for any infinitesimal value. Here we claim that the first timestep is perfect, though, so no errors can pile up. That's odd, right?